# Где собирать логи
1. Ставим Docker desktop
2. Устанавливаем [образ](https://hub.docker.com/r/yandex/clickhouse-server/) Clickhouse
```
!docker run -d -p 0.0.0.0:8123:8123 --volume=/path/to/some/folder/on/disk/some_clickhouse_database:/var/lib/clickhouse --name some-clickhouse-server --ulimit nofile=262144:262144 yandex/clickhouse-server
```

Опция -p 0.0.0.0:8123:8123 открывает доступ к контейнеру по порту 8123 (иногда сразу его нет).

При повторной установке в случае ошибки вида
```
docker: Error response from daemon: Conflict. The container name "/some-clickhouse-server" is already in use by container "34899ff1c1d78111048b762fb730963adac0b90eedb9751f4c5d62aa4d90c589". You have to remove (or rename) that container to be able to reuse that name.
```
удалите контейнер командой (только замените ID контейнера на свой)
```
!docker rm 34899ff1c1d78111048b762fb730963adac0b90eedb9751f4c5d62aa4d90c589
```

Как узнать ID_контейнера
```
!docker ps
```

Как зайти в контейнер (лучше делать в командной строке):
```
docker exec -it ID_контейнера bash
```

Открыть clickhouse-client:
```
docker run -it --rm --link some-clickhouse-server:clickhouse-server yandex/clickhouse-client --host clickhouse-server
```

3. Проверьте наличие доступа к clickhouse в контейнере в браузере, открыв ссылку [localhost:8123](http://localhost:8123), должны увидеть Ok.

4. В примере использовались открытые данные [Метрики](https://clickhouse.tech/docs/ru/getting-started/example-datasets/metrica/).

# Что дает сортировка
Расчет с помощью накопления значений:
- может быть большой расход по памяти
- придется передавать данные между узлами

In [1]:
stats = {}
with open('cards.csv') as f:
    for line in f:
        card, value = line.strip().split(',')

        stats.setdefault(card, 0)
        stats[card] += 1
        
stats

{'червы': 25020, 'пики': 24934, 'бубны': 24932, 'трефы': 25114}

При сортированных данных все обрабатывается построчно

In [1]:
previous_card = None
card_count = 1

with open('cards_sorted.csv') as f:
    f.readline()
    for line in f:
        card, value = line.strip().split(',')

        if previous_card:
            if card == previous_card:
                card_count += 1
            else:
                print(previous_card, card_count)
                card_count = 1

        previous_card = card
        
print(previous_card, card_count)

бубны 24932
пики 24934
трефы 25114
червы 25020


# Загрузка данных в КХ (домашнее задание)
Создайте базу homework
```
CREATE DATABASE homework
```

Создаем таблицу metrika
```
CREATE TABLE homework.metrika
(
    `EventDate` Date,
    `CounterID` UInt32,
    `UserID` UInt64,
    `RegionID` UInt32
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(EventDate)
ORDER BY (CounterID, EventDate, intHash32(UserID))
```

Заливаем данные в таблицу
```
cat metrika_sample.tsv | clickhouse-client --database homework --query "INSERT INTO metrika FORMAT TSV"
```

Посчитайте какой пользователь UserID сделал больше всего просмотров страниц?